In [1]:
import pandas as pd

In [2]:
df_flows_raw = pd.read_excel('outputs_IAEE_2025_run_2021.xlsx', sheet_name='flows_methane_edges')
df_edges_raw = pd.read_excel('inputs_IAEE_2025_run_2021.xlsx', sheet_name='Parameters')

In [3]:
df_flows = pd.DataFrame({
    'Commodity': ['Methane'] * 8,
    'Edge': [
        "('AF_Prod', 'AF')",
        "('AF', 'AF_LNG_exp')",
        "('AF_LNG_exp', 'ES_LNG_imp')",
        "('ES_LNG_imp', 'ES')",
        "('ES', 'FR')",
        "('ES_Prod', 'ES')",
        "('AF_LNG_exp', 'FR_LNG_imp')",
        "('FR_LNG_imp', 'FR')"
    ],
    'Flow': [100, 60, 50, 50, 10, 40, 10, 10],
    'Share': [0.0] * 8
})


In [4]:
df_edges = pd.DataFrame({
    'Commodity': ['Methane'] * 9,
    'Source': [
        'AF_Prod', 'AF', 'AF_LNG_exp', 'ES_LNG_imp', 'ES', 'ES_Prod', 'FR_Prod', 'AF_LNG_exp', 'FR_LNG_imp'
    ],
    'Destination': [
        'AF', 'AF_LNG_exp', 'ES_LNG_imp', 'ES', 'FR', 'ES', 'FR', 'FR_LNG_imp', 'FR'
    ],
    'initial_capacities': [999999] * 9,
    'max_capacities': [999999] * 9,
    'costs_edge': [10, 1000, 1, 2000, 2, 40, 500, 3, 2000],  # From your example
    'new_build_cost': [1e6] * 9
})

In [65]:
df_flows, df_edges = preprocess_data(df_flows, df_edges, commodity='Methane')
flow_graph = build_flow_graph(df_flows)
cost_graph = build_cost_graph(df_edges)
node_flow_data = propagate_costs_optimized(flow_graph, cost_graph)
df_result = aggregate_costs_at_nodes(node_flow_data)
df_result


defaultdict(<class 'list'>, {'AF': [('AF_Prod', 100.0, 10.0, 0)], 'ES': [('ES_Prod', 40.0, 40.0, 0)]})
defaultdict(<class 'list'>, {'AF': [('AF_Prod', 100.0, 10.0, 0)], 'ES': [('ES_Prod', 40.0, 40.0, 0), ('AF_Prod', 50.0, 10.0, 3001.0)], 'AF_LNG_exp': [('AF_Prod', 50.0, 10.0, 1000.0)], 'FR': [('ES_Prod', 10.0, 40.0, 2.0), ('AF_Prod', 10.0, 10.0, 3003.0)], 'ES_LNG_imp': [('AF_Prod', 50.0, 10.0, 1001.0)]})


,Node,TotalFlow,GasCost,TransportCost,AvgPrice
0,AF,100.0,10.000000,0.000000,10.000000
1,ES,90.0,23.333333,1667.222222,1690.555556
2,AF_LNG_exp,50.0,10.000000,1000.000000,1010.000000
3,FR,20.0,25.000000,1502.500000,1527.500000
4,ES_LNG_imp,50.0,10.000000,1001.000000,1011.000000


In [89]:
'''import networkx as nx
import ast

def get_longest_chain_before_nodes(df_flows, commodity):
    # Parse edges and build graph
    df = df_flows[df_flows['Commodity'] == commodity].copy()
    df['Edge'] = df['Edge'].apply(ast.literal_eval)

    G = nx.DiGraph()
    for _, row in df.iterrows():
        src, dst = row['Edge']
        G.add_edge(src, dst)

    if not nx.is_directed_acyclic_graph(G):
        raise ValueError("Graph must be a DAG for longest path calculation.")

    longest_chains = {}
    memo = {}

    def longest_path_to(node):
        if node in memo:
            return memo[node]

        preds = list(G.predecessors(node))
        if not preds:
            # No incoming edge: assume it's a _Prod or disconnected node
            memo[node] = [node]
            return memo[node]

        # Recursively find longest path to each predecessor
        longest_pred_path = max((longest_path_to(p) for p in preds), key=len)
        memo[node] = longest_pred_path + [node]
        return memo[node]

    for node in G.nodes():
        chain = longest_path_to(node)
        # Count edges from first _Prod node in path
        try:
            prod_index = next(i for i, n in enumerate(chain) if n.endswith('_Prod'))
            chain_length = len(chain) - prod_index - 1
        except StopIteration:
            chain_length = len(chain) - 1  # No _Prod, fallback

        longest_chains[node] = {
            'longest_chain_length': max(0, chain_length),
            'longest_chain': chain
        }

    return longest_chains
'''

In [5]:
import pandas as pd
import ast
from collections import defaultdict, deque
import networkx as nx

def preprocess_flows(df_flows, commodity):
    """Parse Edge strings and filter by commodity."""
    df_flows = df_flows[df_flows['Commodity'] == commodity].copy()
    df_flows['Edge'] = df_flows['Edge'].apply(ast.literal_eval)
    return df_flows

def build_flow_and_cost_dicts(df_flows, df_edges, commodity):
    """Build dictionaries for flows and edge costs."""
    flow_dict = defaultdict(float)
    for _, row in df_flows.iterrows():
        flow_dict[row['Edge']] += row['Flow']

    cost_dict = {}
    for _, row in df_edges[df_edges['Commodity'] == commodity].iterrows():
        cost_dict[(row['Source'], row['Destination'])] = row['costs_edge']

    return flow_dict, cost_dict

def get_longest_chain_before_nodes(df_flows, commodity):
    """Compute the longest chain (depth) and path for each node."""
    df = df_flows[df_flows['Commodity'] == commodity].copy()

    G = nx.DiGraph()
    for _, row in df.iterrows():
        src, dst = row['Edge']
        G.add_edge(src, dst)

    if not nx.is_directed_acyclic_graph(G):
        raise ValueError("Graph must be a DAG for longest path calculation.")

    longest_chains = {}
    memo = {}

    def longest_path_to(node):
        if node in memo:
            return memo[node]
        preds = list(G.predecessors(node))
        if not preds:
            memo[node] = [node]
            return memo[node]
        longest_pred_path = max((longest_path_to(p) for p in preds), key=len)
        memo[node] = longest_pred_path + [node]
        return memo[node]

    for node in G.nodes():
        chain = longest_path_to(node)
        try:
            prod_index = next(i for i, n in enumerate(chain) if n.endswith('_Prod'))
            chain_length = len(chain) - prod_index - 1
        except StopIteration:
            chain_length = len(chain) - 1
        longest_chains[node] = {
            'longest_chain_length': max(0, chain_length),
            'longest_chain': chain
        }

    return longest_chains

def propagate_gas_mix(flow_dict, cost_dict, chain_info):
    """Propagate gas and transport costs through the network by chain level."""
    node_inflows = defaultdict(list)
    max_depth = max(info['longest_chain_length'] for info in chain_info.values())

    # First: add all _Prod flows
    for (src, dst), flow in flow_dict.items():
        if src.endswith('_Prod'):
            cost = cost_dict.get((src, dst), 0)
            node_inflows[dst].append({
                'origin': src,
                'flow': flow,
                'gas_cost': cost,
                'transport_cost': 0
            })

    # Process by depth
    for depth in range(1, max_depth + 1):
        nodes_at_depth = [node for node, info in chain_info.items() if info['longest_chain_length'] == depth]

        for node in nodes_at_depth:
            inflows = node_inflows[node]
            total_inflow = sum(i['flow'] for i in inflows)
            if total_inflow == 0:
                continue

            for (src, dst), flow in flow_dict.items():
                if src != node:
                    continue

                for entry in inflows:
                    share = entry['flow'] / total_inflow if total_inflow > 0 else 0
                    proportional_flow = share * flow
                    cost = cost_dict.get((src, dst), 0)

                    node_inflows[dst].append({
                        'origin': entry['origin'],
                        'flow': proportional_flow,
                        'gas_cost': entry['gas_cost'],
                        'transport_cost': entry['transport_cost'] + cost
                    })

    return node_inflows

def aggregate_node_costs(node_inflows):
    """Aggregate weighted gas and transport costs per node."""
    results = []

    for node, inflows in node_inflows.items():
        total_flow = sum(i['flow'] for i in inflows)
        if total_flow == 0:
            continue

        weighted_gas_cost = sum(i['flow'] * i['gas_cost'] for i in inflows) / total_flow
        weighted_transport_cost = sum(i['flow'] * i['transport_cost'] for i in inflows) / total_flow

        results.append({
            'Node': node,
            'Total Flow': total_flow,
            'Gas Cost': weighted_gas_cost,
            'Transport Cost': weighted_transport_cost,
            'Total Cost': weighted_gas_cost + weighted_transport_cost
        })

    return pd.DataFrame(results).sort_values('Node').reset_index(drop=True)

def compute_average_costs(df_flows, df_edges, commodity='Methane'):
    """Main function to compute gas and transport costs per node."""
    df_flows = preprocess_flows(df_flows, commodity)
    flow_dict, cost_dict = build_flow_and_cost_dicts(df_flows, df_edges, commodity)
    chain_info = get_longest_chain_before_nodes(df_flows, commodity)
    node_inflows = propagate_gas_mix(flow_dict, cost_dict, chain_info)
    return aggregate_node_costs(node_inflows)


In [6]:
results = compute_average_costs(df_flows, df_edges, commodity='Methane')
results

,Node,Total Flow,Gas Cost,Transport Cost,Total Cost
0,AF,100.0,10.000000,0.000000,10.000000
1,AF_LNG_exp,60.0,10.000000,1000.000000,1010.000000
2,ES,90.0,23.333333,1667.222222,1690.555556
3,ES_LNG_imp,50.0,10.000000,1001.000000,1011.000000
4,FR,20.0,16.666667,2336.111111,2352.777778
5,FR_LNG_imp,10.0,10.000000,1003.000000,1013.000000


In [7]:
import pandas as pd
import ast
from collections import defaultdict, deque

def preprocess_flows(df_flows, commodity):
    df_flows = df_flows[df_flows['Commodity'] == commodity].copy()
    df_flows['Edge'] = df_flows['Edge'].apply(ast.literal_eval)
    return df_flows

def build_flow_and_cost_dicts(df_flows, df_edges, commodity):
    flow_dict = defaultdict(float)
    for _, row in df_flows.iterrows():
        flow_dict[row['Edge']] += row['Flow']

    cost_dict = {}
    for _, row in df_edges[df_edges['Commodity'] == commodity].iterrows():
        cost_dict[(row['Source'], row['Destination'])] = row['costs_edge']

    return flow_dict, cost_dict

def compute_longest_chain_lengths(flow_dict):
    predecessors = defaultdict(list)
    for (src, dst) in flow_dict:
        predecessors[dst].append(src)

    longest_paths = {}

    def dfs(node):
        if node in longest_paths:
            return longest_paths[node]
        max_depth = 0
        for pred in predecessors[node]:
            if pred.endswith('_Prod'):
                depth = 1
            else:
                depth = dfs(pred) + 1
            max_depth = max(max_depth, depth)
        longest_paths[node] = max_depth
        return max_depth

    all_nodes = set([dst for _, dst in flow_dict.keys()] + [src for src, _ in flow_dict.keys()])
    for node in all_nodes:
        if node not in longest_paths:
            dfs(node)

    return longest_paths

def propagate_gas_mix_by_level(flow_dict, cost_dict, longest_paths):
    node_inflows = defaultdict(list)

    max_level = max(longest_paths.values())

    for (src, dst), flow in flow_dict.items():
        if src.endswith('_Prod'):
            cost = cost_dict.get((src, dst), 0)
            node_inflows[dst].append({
                'origin': src,
                'flow': flow,
                'gas_cost': cost,
                'transport_cost': 0
            })

    for level in range(1, max_level + 1):
        for node, node_level in longest_paths.items():
            if node_level != level:
                continue

            inflows = node_inflows.get(node, [])
            total_inflow = sum(i['flow'] for i in inflows)
            if total_inflow == 0:
                continue

            for (src, dst), flow in flow_dict.items():
                if src != node:
                    continue

                cost = cost_dict.get((src, dst), 0)
                for entry in inflows:
                    share = entry['flow'] / total_inflow if total_inflow > 0 else 0
                    proportional_flow = share * flow

                    node_inflows[dst].append({
                        'origin': entry['origin'],
                        'flow': proportional_flow,
                        'gas_cost': entry['gas_cost'],
                        'transport_cost': entry['transport_cost'] + cost
                    })

    return node_inflows

def aggregate_node_costs(node_inflows):
    results = []

    for node, inflows in node_inflows.items():
        total_flow = sum(i['flow'] for i in inflows)
        if total_flow == 0:
            continue

        weighted_gas_cost = sum(i['flow'] * i['gas_cost'] for i in inflows) / total_flow
        weighted_transport_cost = sum(i['flow'] * i['transport_cost'] for i in inflows) / total_flow

        results.append({
            'Node': node,
            'Total Flow': total_flow,
            'Gas Cost': weighted_gas_cost,
            'Transport Cost': weighted_transport_cost,
            'Total Cost': weighted_gas_cost + weighted_transport_cost
        })

    return pd.DataFrame(results).sort_values('Node').reset_index(drop=True)

def compute_source_shares(node_inflows):
    """Compute source shares per destination node."""
    share_data = []

    for node, inflows in node_inflows.items():
        total_flow = sum(i['flow'] for i in inflows)
        if total_flow == 0:
            continue

        origin_flows = defaultdict(float)
        for i in inflows:
            origin_flows[i['origin']] += i['flow']

        for origin, flow in origin_flows.items():
            share_data.append({
                'Node': node,
                'Origin': origin,
                'Flow': flow,
                'Share': flow / total_flow
            })

    return pd.DataFrame(share_data).sort_values(['Node', 'Origin']).reset_index(drop=True)

def compute_average_costs(df_flows, df_edges, commodity='Methane'):
    df_flows = preprocess_flows(df_flows, commodity)
    flow_dict, cost_dict = build_flow_and_cost_dicts(df_flows, df_edges, commodity)
    longest_paths = compute_longest_chain_lengths(flow_dict)
    node_inflows = propagate_gas_mix_by_level(flow_dict, cost_dict, longest_paths)
    return aggregate_node_costs(node_inflows), compute_source_shares(node_inflows)


In [8]:
df_costs, df_source_shares = compute_average_costs(df_flows, df_edges, commodity='Methane')

In [9]:
df_costs

,Node,Total Flow,Gas Cost,Transport Cost,Total Cost
0,AF,100.0,10.000000,0.000000,10.000000
1,AF_LNG_exp,60.0,10.000000,1000.000000,1010.000000
2,ES,90.0,23.333333,1667.222222,1690.555556
3,ES_LNG_imp,50.0,10.000000,1001.000000,1011.000000
4,FR,20.0,16.666667,2336.111111,2352.777778
5,FR_LNG_imp,10.0,10.000000,1003.000000,1013.000000


In [11]:
# Keep only nodes with no underscore (pure region names)
df_costs_cleaned = df_costs[~df_costs['Node'].str.contains('_')].copy()

# Drop the 'Total Flow' column
df_costs_cleaned = df_costs_cleaned.drop(columns=['Total Flow'])

# Reset index
df_costs_cleaned = df_costs_cleaned.reset_index(drop=True)

# Display result
df_costs_cleaned


,Node,Gas Cost,Transport Cost,Total Cost
0,AF,10.000000,0.000000,10.000000
1,ES,23.333333,1667.222222,1690.555556
2,FR,16.666667,2336.111111,2352.777778


In [10]:
df_source_shares

,Node,Origin,Flow,Share
0,AF,AF_Prod,100.000000,1.000000
1,AF_LNG_exp,AF_Prod,60.000000,1.000000
2,ES,AF_Prod,50.000000,0.555556
3,ES,ES_Prod,40.000000,0.444444
4,ES_LNG_imp,AF_Prod,50.000000,1.000000
5,FR,AF_Prod,15.555556,0.777778
6,FR,ES_Prod,4.444444,0.222222
7,FR_LNG_imp,AF_Prod,10.000000,1.000000


In [13]:
# Keep only rows where 'Node' has no underscore (i.e., pure country nodes)
df_shares_cleaned = df_source_shares[~df_source_shares['Node'].str.contains('_')].copy()

# Drop the 'Flow' column
df_shares_cleaned = df_shares_cleaned.drop(columns=['Flow'])

# Reset index
df_shares_cleaned = df_shares_cleaned.reset_index(drop=True)

# Display result
df_shares_cleaned

,Node,Origin,Share
0,AF,AF_Prod,1.000000
1,ES,AF_Prod,0.555556
2,ES,ES_Prod,0.444444
3,FR,AF_Prod,0.777778
4,FR,ES_Prod,0.222222


In [114]:
df_flows_raw = df_flows_raw[df_flows_raw['Flow'] != 0].reset_index(drop=True)

In [115]:
df_result  = compute_average_costs(df_flows_raw, df_edges_raw, commodity='Methane')

In [116]:
df_result

,Node,Total Flow,Gas Cost,Transport Cost,Total Cost
0,AF,7.563890e+05,15948.349277,0.000000,15948.349277
1,AF_LNG_exp,3.611996e+05,15948.349277,8530.353700,24478.702977
2,AL,8.404858e+04,11593.467101,10537.874906,22131.342007
3,AS,1.854224e+06,30963.332090,2012.438577,32975.770666
4,AS_LNG_imp,3.399022e+05,23339.047722,8760.299360,32099.347082
...,...,...,...,...,...
76,UA,2.877569e+05,18774.605159,279068.749192,297843.354351
77,UK,8.835563e+05,21311.207073,6288.061139,27599.268212
78,UK_LNG_imp,4.699370e+05,22950.063594,9134.819306,32084.882899
79,USA,9.035422e+06,22950.063594,0.000000,22950.063594


In [117]:
df_result.to_excel('test_avg.xlsx')